# 01 — Паспорт исходных данных

Этот ноутбук формирует только общую информацию о файлах для `docs/01_data.md`: пути, роли, размеры, схему и SHA-256. Он не анализирует распределения, пропуски или target.

Конфигурация файлов, key, target и описаний столбцов хранится централизованно в `src/ml_project/config.py`.

In [ ]:
from pathlib import Path
import sys

from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate
    for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").exists() and (candidate / "src").exists()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project import (
    DataCatalog,
    DatasetProfiler,
    MarkdownDocument,
    build_data_blocks,
    build_eda_blocks,
    build_field_descriptions_template,
)
from ml_project.config import (
    DATASETS,
    FIELD_DESCRIPTIONS,
    INFERENCE_DATASET,
    KEY,
    RAW_DIR,
    TARGET,
    TRAIN_DATASET,
)

print(f"Корень проекта: {PROJECT_ROOT}")

In [ ]:
catalog = DataCatalog(PROJECT_ROOT, RAW_DIR, DATASETS)
catalog.validate()
datasets = catalog.load_all()

print("Загружены наборы:", ", ".join(datasets))

## 1. Реестр файлов

In [ ]:
file_report = catalog.file_report()
display(
    file_report.drop(columns=["sha256"]).style.format(
        {"disk_kib": "{:.1f}", "memory_mib": "{:.3f}"}
    )
)

## 2. Схема и роли столбцов

In [ ]:
schema_report = catalog.schema_report(
    key=KEY,
    target=TARGET,
    inference_dataset=INFERENCE_DATASET,
    field_descriptions=FIELD_DESCRIPTIONS,
)
display(schema_report)

## 3. Версия файлов

In [ ]:
version_report = file_report[["dataset", "file", "sha256"]]
display(version_report)

## 4. Заготовка описаний полей

Ячейка автоматически находит все столбцы во входных файлах и печатает готовый словарь `FIELD_DESCRIPTIONS`. Уже заполненные описания сохраняются, новые поля получают пустую строку. `config.py` не изменяется автоматически: скопируйте заготовку только после проверки.


In [ ]:
description_template = build_field_descriptions_template(
    catalog,
    FIELD_DESCRIPTIONS,
)
unconfigured_fields = list(
    dict.fromkeys(
        str(field)
        for dataset in datasets.values()
        for field in dataset.columns
        if not str(FIELD_DESCRIPTIONS.get(str(field), "")).strip()
    )
)

if unconfigured_fields:
    print(
        f"Без описания: {len(unconfigured_fields)}. "
        "Заполните пустые строки перед синхронизацией документа."
    )
else:
    print("Все найденные поля уже имеют описание.")

print("\nГотовая заготовка для src/ml_project/config.py:\n")
print(description_template)


## 5. Синхронизация с документом

Следующая ячейка изменяет только блоки между HTML-маркерами `auto:*` в `docs/01_data.md`. Ручной текст документа сохраняется.

In [ ]:
data_blocks = build_data_blocks(
    catalog,
    key=KEY,
    target=TARGET,
    inference_dataset=INFERENCE_DATASET,
    field_descriptions=FIELD_DESCRIPTIONS,
)
updated_blocks = MarkdownDocument(
    PROJECT_ROOT / "docs" / "01_data.md"
).update_blocks(data_blocks)

print("Обновлены блоки:", ", ".join(updated_blocks))

## 6. Следующий шаг

После синхронизации проверьте `docs/01_data.md`, вручную заполните источник, правила доступа и ограничения. Когда Stage Gate выполнен, переходите к `notebooks/02_eda.ipynb`.